# Let's go PRO!

Advanced RAG Techniques!

Let's start by digging into ingest:

1. No LangChain! Just native for maximum flexibility
2. Let's use an LLM to divide up chunks in a sensible way
3. Let's use the best chunk size and encoder from yesterday
4. Let's also have the LLM rewrite chunks in a way that's most useful ("document pre-processing")

In [3]:
from pathlib import Path
from openai import OpenAI
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from chromadb import PersistentClient
from tqdm import tqdm
from litellm import completion
import numpy as np
from sklearn.manifold import TSNE
import plotly.graph_objects as go
from IPython.display import Markdown, display


load_dotenv(override=True)

MODEL = "gpt-4.1-mini"

DB_NAME = "preprocessed_db"
collection_name = "docs"
embedding_model = "text-embedding-3-large"
KNOWLEDGE_BASE_PATH = Path("knowledge-base")
AVERAGE_CHUNK_SIZE = 500

openai = OpenAI()

In [4]:
# Inspired by LangChain's Document - let's have something similar

class Result(BaseModel):
    page_content: str
    metadata: dict

In [5]:
# A class to perfectly represent a chunk

class Chunk(BaseModel):
    headline: str = Field(description="A brief heading for this chunk, typically a few words, that is most likely to be surfaced in a query")
    summary: str = Field(description="A few sentences summarizing the content of this chunk to answer common questions")
    original_text: str = Field(description="The original text of this chunk from the provided document, exactly as is, not changed in any way")

    def as_result(self, document):
        metadata = {"source": document["source"], "type": document["type"]}
        return Result(page_content=self.headline + "\n\n" + self.summary + "\n\n" + self.original_text,metadata=metadata)


class Chunks(BaseModel):
    chunks: list[Chunk]

## Three steps:

1. Fetch documents from the knowledge base, like LangChain did
2. Call an LLM to turn documents into Chunks
3. Store the Chunks in Chroma

That's it!

### Let's start with Step 1

In [6]:
def fetch_documents():
    """A homemade version of the LangChain DirectoryLoader"""

    documents = []

    for folder in KNOWLEDGE_BASE_PATH.iterdir():
        doc_type = folder.name
        for file in folder.rglob("*.md"):
            with open(file, "r", encoding="utf-8") as f:
                documents.append({"type": doc_type, "source": file.as_posix(), "text": f.read()})

    print(f"Loaded {len(documents)} documents")
    return documents

In [7]:
documents = fetch_documents()

Loaded 76 documents


In [8]:
documents[0]

{'type': 'products',
 'source': 'knowledge-base/products/Rellm.md',
 'text': "# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.\n\n## Features\n\n### AI-Driven Analytics\nRellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.\n\n### Seamless Integ

In [9]:
display(Markdown(documents[0]["text"]))

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.

### Seamless Integrations
Rellm's architecture is designed for effortless integration with existing systems. Whether it's policy management, claims processing, or financial reporting, Rellm connects seamlessly with diverse data sources to create a unified ecosystem.

### Risk Assessment Module
The comprehensive risk assessment module within Rellm allows insurers to evaluate risk profiles accurately. By leveraging historical data and advanced modeling techniques, Rellm provides a clear picture of potential liabilities and expected outcomes.

### Customizable Dashboard
Rellm features a customizable dashboard that presents key metrics and performance indicators in an intuitive interface. Users can tailor their view to focus on what matters most to their business, enhancing user experience and productivity.

### Regulatory Compliance Tools
Rellm includes built-in compliance tracking features to help organizations meet local and international regulatory standards. This ensures that reinsurance practices remain transparent and accountable.

### Client and Broker Portals
Rellm offers dedicated portals for both clients and brokers, facilitating real-time communication and documentation sharing. This strengthens partnerships and drives operational excellence across the board.

## Pricing

Insurellm offers flexible pricing plans for Rellm to cater to various business needs:

- **Basic Plan**: $5,000/month
  - Includes access to core features and standard integrations.
  
- **Professional Plan**: $10,000/month
  - Includes all features, advanced integrations, and priority customer support.
  
- **Enterprise Plan**: Custom pricing
  - Tailored solutions with personalized features, extensive integrations, and dedicated account management.

Join the growing number of organizations leveraging Rellm to enhance their reinsurance processes while driving profitability and compliance. 

## 2025-2026 Roadmap

At Insurellm, we are committed to the continuous improvement of Rellm. Our roadmap for 2025-2026 includes:

- **Q3 2025**: 
  - Launch of the Rellm Mobile App for on-the-go insights and management.
  - Introduction of augmented reality (AR) features for interactive risk assessments.

- **Q1 2026**: 
  - Deployment of advanced machine learning models for even more accurate risk predictions.
  - Expansion of integration capabilities to support emerging technologies in the insurance sector.

- **Q3 2026**: 
  - Release of a community platform for Rellm users to exchange insights, tips, and best practices.
  - Launch of Rellm 2.0, featuring enhanced user interface and premium features based on user feedback.

Experience the future of reinsurance with Rellm, where innovation meets reliability. Let Insurellm help you navigate the complexities of the reinsurance market smarter and faster.

In [10]:
how_many = (len(documents[0]["text"]) // AVERAGE_CHUNK_SIZE) + 1
how_many

8

### Donezo! On to Step 2 - make the chunks

In [11]:
def make_prompt(document):
    how_many = (len(document["text"]) // AVERAGE_CHUNK_SIZE) + 1
    return f"""
You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: {document["type"]}
The document has been retrieved from: {document["source"]}

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into {how_many} chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

{document["text"]}

Respond with the chunks.
"""

In [12]:
display(Markdown(make_prompt(documents[0])))


You take a document and you split the document into overlapping chunks for a KnowledgeBase.

The document is from the shared drive of a company called Insurellm.
The document is of type: products
The document has been retrieved from: knowledge-base/products/Rellm.md

A chatbot will use these chunks to answer questions about the company.
You should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.
This document should probably be split into 8 chunks, but you can have more or less as appropriate.
There should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.

For each chunk, you should provide a headline, a summary, and the original text of the chunk.
Together your chunks should represent the entire document with overlap.

Here is the document:

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.

### Seamless Integrations
Rellm's architecture is designed for effortless integration with existing systems. Whether it's policy management, claims processing, or financial reporting, Rellm connects seamlessly with diverse data sources to create a unified ecosystem.

### Risk Assessment Module
The comprehensive risk assessment module within Rellm allows insurers to evaluate risk profiles accurately. By leveraging historical data and advanced modeling techniques, Rellm provides a clear picture of potential liabilities and expected outcomes.

### Customizable Dashboard
Rellm features a customizable dashboard that presents key metrics and performance indicators in an intuitive interface. Users can tailor their view to focus on what matters most to their business, enhancing user experience and productivity.

### Regulatory Compliance Tools
Rellm includes built-in compliance tracking features to help organizations meet local and international regulatory standards. This ensures that reinsurance practices remain transparent and accountable.

### Client and Broker Portals
Rellm offers dedicated portals for both clients and brokers, facilitating real-time communication and documentation sharing. This strengthens partnerships and drives operational excellence across the board.

## Pricing

Insurellm offers flexible pricing plans for Rellm to cater to various business needs:

- **Basic Plan**: $5,000/month
  - Includes access to core features and standard integrations.
  
- **Professional Plan**: $10,000/month
  - Includes all features, advanced integrations, and priority customer support.
  
- **Enterprise Plan**: Custom pricing
  - Tailored solutions with personalized features, extensive integrations, and dedicated account management.

Join the growing number of organizations leveraging Rellm to enhance their reinsurance processes while driving profitability and compliance. 

## 2025-2026 Roadmap

At Insurellm, we are committed to the continuous improvement of Rellm. Our roadmap for 2025-2026 includes:

- **Q3 2025**: 
  - Launch of the Rellm Mobile App for on-the-go insights and management.
  - Introduction of augmented reality (AR) features for interactive risk assessments.

- **Q1 2026**: 
  - Deployment of advanced machine learning models for even more accurate risk predictions.
  - Expansion of integration capabilities to support emerging technologies in the insurance sector.

- **Q3 2026**: 
  - Release of a community platform for Rellm users to exchange insights, tips, and best practices.
  - Launch of Rellm 2.0, featuring enhanced user interface and premium features based on user feedback.

Experience the future of reinsurance with Rellm, where innovation meets reliability. Let Insurellm help you navigate the complexities of the reinsurance market smarter and faster.

Respond with the chunks.


In [13]:
def make_messages(document):
    return [
        {"role": "user", "content": make_prompt(document)},
    ]

In [14]:
make_messages(documents[0])

[{'role': 'user',
  'content': "\nYou take a document and you split the document into overlapping chunks for a KnowledgeBase.\n\nThe document is from the shared drive of a company called Insurellm.\nThe document is of type: products\nThe document has been retrieved from: knowledge-base/products/Rellm.md\n\nA chatbot will use these chunks to answer questions about the company.\nYou should divide up the document as you see fit, being sure that the entire document is returned in the chunks - don't leave anything out.\nThis document should probably be split into 8 chunks, but you can have more or less as appropriate.\nThere should be overlap between the chunks as appropriate; typically about 25% overlap or about 50 words, so you have the same text in multiple chunks for best retrieval results.\n\nFor each chunk, you should provide a headline, a summary, and the original text of the chunk.\nTogether your chunks should represent the entire document with overlap.\n\nHere is the document:\n\n#

In [16]:
def process_document(document):
    messages = make_messages(document)
    response = completion(model=MODEL, messages=messages, response_format=Chunks)
    reply = response.choices[0].message.content
    doc_as_chunks = Chunks.model_validate_json(reply).chunks
    return [chunk.as_result(document) for chunk in doc_as_chunks]

In [17]:
doc_chunks = process_document(documents[0])
len(doc_chunks)

8

In [18]:
doc_chunks

[Result(page_content='Introduction to Rellm and its Purpose\n\nRellm is an AI-powered enterprise reinsurance product developed by Insurellm aimed at revolutionizing the reinsurance industry. It provides advanced risk management capabilities, enhances decision-making, and improves operational efficiencies by leveraging artificial intelligence and robust analytics.\n\n# Product Summary\n\n# Rellm: AI-Powered Enterprise Reinsurance Solution\n\n## Summary\n\nRellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.', metadata={'source': '

In [19]:
doc_chunks[0].metadata


{'source': 'knowledge-base/products/Rellm.md', 'type': 'products'}

In [21]:
display(Markdown(doc_chunks[0].page_content))

Introduction to Rellm and its Purpose

Rellm is an AI-powered enterprise reinsurance product developed by Insurellm aimed at revolutionizing the reinsurance industry. It provides advanced risk management capabilities, enhances decision-making, and improves operational efficiencies by leveraging artificial intelligence and robust analytics.

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

In [22]:
display(Markdown(doc_chunks[1].page_content))

AI-Driven Analytics Feature

Rellm leverages cutting-edge AI algorithms to offer predictive insights into risk exposures, enabling forecasting and informed decision-making. This real-time data analysis empowers reinsurance professionals by providing actionable intelligence.

With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

## Features

### AI-Driven Analytics
Rellm utilizes cutting-edge AI algorithms to provide predictive insights into risk exposures, enabling users to forecast trends and make informed decisions. Its real-time data analysis empowers reinsurance professionals with actionable intelligence.

In [24]:
display(Markdown(doc_chunks[-2].page_content))

2025-2026 Roadmap: Upcoming Features Q3 2025 to Q1 2026

Insurellm is committed to enhancing Rellm through their 2025-2026 roadmap. Planned features include the Q3 2025 launch of a mobile app and augmented reality capabilities for interactive risk assessments, followed in Q1 2026 by advanced machine learning models for improved risk prediction and expanded integration support for emerging insurance technologies.

## 2025-2026 Roadmap

At Insurellm, we are committed to the continuous improvement of Rellm. Our roadmap for 2025-2026 includes:

- **Q3 2025**: 
  - Launch of the Rellm Mobile App for on-the-go insights and management.
  - Introduction of augmented reality (AR) features for interactive risk assessments.

- **Q1 2026**: 
  - Deployment of advanced machine learning models for even more accurate risk predictions.
  - Expansion of integration capabilities to support emerging technologies in the insurance sector.

In [23]:
display(Markdown(doc_chunks[-1].page_content))

2025-2026 Roadmap: Community Platform and Rellm 2.0 Release

Later in the 2025-2026 roadmap, specifically Q3 2026, Insurellm plans to release a community platform for Rellm users to share knowledge and best practices. Additionally, Rellm 2.0 will launch featuring an enhanced user interface and premium features inspired by user feedback, continuing the innovation and reliability in reinsurance management.

- **Q3 2026**: 
  - Release of a community platform for Rellm users to exchange insights, tips, and best practices.
  - Launch of Rellm 2.0, featuring enhanced user interface and premium features based on user feedback.

Experience the future of reinsurance with Rellm, where innovation meets reliability. Let Insurellm help you navigate the complexities of the reinsurance market smarter and faster.

In [25]:
def create_chunks(documents):
    chunks = []
    for doc in tqdm(documents):
        chunks.extend(process_document(doc))
    return chunks

In [26]:
chunks = create_chunks(documents)

100%|██████████| 76/76 [29:21<00:00, 23.18s/it]


In [27]:
print(len(chunks))

644


In [28]:
display(Markdown(chunks[0].page_content))

Introduction and Product Overview

Rellm is an AI-powered enterprise reinsurance product developed by Insurellm aimed at transforming operations in the reinsurance industry. It leverages artificial intelligence to enhance risk management, decision-making, and operational efficiencies. The platform integrates seamlessly with existing systems and uses robust analytics to help insurers manage portfolios proactively and respond agilely to market changes.

# Product Summary

# Rellm: AI-Powered Enterprise Reinsurance Solution

## Summary

Rellm is an innovative enterprise reinsurance product developed by Insurellm, designed to transform the way reinsurance companies operate. Harnessing the power of artificial intelligence, Rellm offers an advanced platform that redefines risk management, enhances decision-making processes, and optimizes operational efficiencies within the reinsurance industry. With seamless integrations and robust analytics, Rellm enables insurers to proactively manage their portfolios and respond to market dynamics with agility.

In [30]:
display(Markdown(chunks[-2].page_content))

Compensation History and Current Earnings

Brandon's compensation has steadily increased since joining Insurellm, starting with a base salary of $58,000 in 2021 and reaching $62,000 plus a $1,500 bonus in 2023. In 2022, he did not receive a bonus due to performance issues. His salary reflects his evolving role and performance within the company.

## Compensation History
- **2023:** Base Salary: $62,000 + Bonus: $1,500
- **2022:** Base Salary: $60,000 + Bonus: $0 (no bonus due to performance)
- **2021:** Base Salary: $58,000 (started mid-year)

## Other HR Notes
- **Education:** Associate Degree in Information Technology from Phoenix Community College

In [29]:
display(Markdown(chunks[-1].page_content))

Education, Skills, and Development Plans

Brandon holds an Associate Degree in Information Technology and a CompTIA A+ certification and is working towards Network+. Currently on a 90-day Performance Improvement Plan since August 2023 focusing on improving response times and customer communication skills. His technical skills include system troubleshooting, SQL basics, and API debugging, while soft skills and time management need improvement. Management is supporting his development in structured troubleshooting and empathetic communication.

## Other HR Notes
- **Education:** Associate Degree in Information Technology from Phoenix Community College
- **Certifications:** CompTIA A+, working toward Network+ certification
- **Performance Improvement Plan:** Currently on 90-day PIP (started August 2023) focusing on response time improvements and customer communication skills
- **Skills:** Strong in system troubleshooting, SQL basics, and API debugging. Needs improvement in soft skills and time management.
- **Development:** Working with manager on structured troubleshooting approach and empathetic customer communication
- **Feedback:** Technically competent but struggles under pressure. Tends to focus on technical details rather than customer experience. Improving but needs consistent focus.

### Well that was easy! If a bit slow.

In the python module version, I sneakily use the multi-processing Pool to run this in parallel,
but if you get a Rate Limit Error you can turn this off in the code.

### Finally, Step 3 - save the embeddings

In [31]:
def create_embeddings(chunks):
    chroma = PersistentClient(path=DB_NAME)
    if collection_name in [c.name for c in chroma.list_collections()]:
        chroma.delete_collection(collection_name)

    texts = [chunk.page_content for chunk in chunks]
    emb = openai.embeddings.create(model=embedding_model, input=texts).data
    vectors = [e.embedding for e in emb]

    collection = chroma.get_or_create_collection(collection_name)

    ids = [str(i) for i in range(len(chunks))]
    metas = [chunk.metadata for chunk in chunks]

    collection.add(ids=ids, embeddings=vectors, documents=texts, metadatas=metas)
    print(f"Vectorstore created with {collection.count()} documents")

In [32]:
create_embeddings(chunks)

Vectorstore created with 644 documents


# Nothing more to do here... right?

Wait! Didja think I'd forget??

In [33]:
chroma = PersistentClient(path=DB_NAME)
collection = chroma.get_or_create_collection(collection_name)
result = collection.get(include=['embeddings', 'documents', 'metadatas'])
vectors = np.array(result['embeddings'])
documents = result['documents']
metadatas = result['metadatas']
doc_types = [metadata['type'] for metadata in metadatas]
colors = [['blue', 'green', 'red', 'orange'][['products', 'employees', 'contracts', 'company'].index(t)] for t in doc_types]

In [34]:
tsne = TSNE(n_components=2, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 2D scatter plot
fig = go.Figure(data=[go.Scatter(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(title='2D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x',yaxis_title='y'),
    width=800,
    height=600,
    margin=dict(r=20, b=10, l=10, t=40)
)

fig.show()

In [35]:
tsne = TSNE(n_components=3, random_state=42)
reduced_vectors = tsne.fit_transform(vectors)

# Create the 3D scatter plot
fig = go.Figure(data=[go.Scatter3d(
    x=reduced_vectors[:, 0],
    y=reduced_vectors[:, 1],
    z=reduced_vectors[:, 2],
    mode='markers',
    marker=dict(size=5, color=colors, opacity=0.8),
    text=[f"Type: {t}<br>Text: {d[:100]}..." for t, d in zip(doc_types, documents)],
    hoverinfo='text'
)])

fig.update_layout(
    title='3D Chroma Vector Store Visualization',
    scene=dict(xaxis_title='x', yaxis_title='y', zaxis_title='z'),
    width=900,
    height=700,
    margin=dict(r=10, b=10, l=10, t=40)
)

fig.show()

## And now - let's build an Advanced RAG!

We will use these techniques:

1. Reranking - reorder the rank results
2. Query re-writing

In [36]:
class RankOrder(BaseModel):
    order: list[int] = Field(
        description="The order of relevance of chunks, from most relevant to least relevant, by chunk id number"
    )

In [37]:
def rerank(question, chunks):
    system_prompt = """
You are a document re-ranker.
You are provided with a question and a list of relevant chunks of text from a query of a knowledge base.
The chunks are provided in the order they were retrieved; this should be approximately ordered by relevance, but you may be able to improve on that.
You must rank order the provided chunks by relevance to the question, with the most relevant chunk first.
Reply only with the list of ranked chunk ids, nothing else. Include all the chunk ids you are provided with, reranked.
"""
    user_prompt = f"The user has asked the following question:\n\n{question}\n\nOrder all the chunks of text by relevance to the question, from most relevant to least relevant. Include all the chunk ids you are provided with, reranked.\n\n"
    user_prompt += "Here are the chunks:\n\n"
    for index, chunk in enumerate(chunks):
        user_prompt += f"# CHUNK ID: {index + 1}:\n\n{chunk.page_content}\n\n"
    user_prompt += "Reply only with the list of ranked chunk ids, nothing else."
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    response = completion(model=MODEL, messages=messages, response_format=RankOrder)
    reply = response.choices[0].message.content
    order = RankOrder.model_validate_json(reply).order
    print(order)
    return [chunks[i - 1] for i in order]

In [38]:
RETRIEVAL_K = 10

def fetch_context_unranked(question):
    query = openai.embeddings.create(model=embedding_model, input=[question]).data[0].embedding
    results = collection.query(query_embeddings=[query], n_results=RETRIEVAL_K)
    chunks = []
    for result in zip(results["documents"][0], results["metadatas"][0]):
        chunks.append(Result(page_content=result[0], metadata=result[1]))
    return chunks

In [39]:
question = "Who won the IIOTY award?"
chunks = fetch_context_unranked(question)

In [43]:
for chunk in chunks:
    print(chunk.page_content[:25]+"...")

Senior Data Engineer and ...
Additional HR Notes and F...
Leadership Training and 2...
Annual Performance Histor...
Annual Performance Histor...
Annual Performance Histor...
Career Advancement and Pe...
Career Progression at Ins...
Annual Performance Highli...
Recognition, Mentorship, ...


In [44]:
reranked = rerank(question, chunks)

[1, 2, 10, 3, 7, 9, 4, 5, 6, 8]


In [45]:
for chunk in reranked:
    print(chunk.page_content[:25]+"...")

Senior Data Engineer and ...
Additional HR Notes and F...
Recognition, Mentorship, ...
Leadership Training and 2...
Career Advancement and Pe...
Annual Performance Highli...
Annual Performance Histor...
Annual Performance Histor...
Annual Performance Histor...
Career Progression at Ins...


In [46]:
question = "Who went to Manchester University?"
RETRIEVAL_K = 20
chunks = fetch_context_unranked(question)
for index, c in enumerate(chunks):
    if "manchester" in c.page_content.lower():
        print(index)

16


In [50]:
display(Markdown(chunks[0].page_content))

Early Career: Junior Data Engineer at Insurellm

Maxine started at Insurellm in January 2017 as a Junior Data Engineer focusing on ETL processes and data integration. She learned the company's data architecture quickly and collaborated with the team to streamline workflows, working until October 2018 in this role.

## Insurellm Career Progression
- **January 2017 - October 2018**: **Junior Data Engineer**  
  * Maxine joined Insurellm as a Junior Data Engineer, focusing primarily on ETL processes and data integration tasks. She quickly learned Insurellm's data architecture, collaborating with other team members to streamline data workflows.  

In [51]:
display(Markdown(chunks[16].page_content))

Compensation and Professional Development

Jessica's base salary has increased from $68,000 in 2020 to $92,000 in 2023, with annual bonuses increasing to $6,000 in 2023. She holds a BS in Computer Science from the University of Manchester and is proficient in React, TypeScript, HTML/CSS, and Jest. She is currently learning Next.js and GraphQL and completed an Advanced React Patterns course in 2023. She also actively contributes to open-source projects.

## Compensation History
- **2023:** Base Salary: $92,000 + Bonus: $6,000
- **2022:** Base Salary: $85,000 + Bonus: $4,000
- **2021:** Base Salary: $72,000 + Bonus: $2,000
- **2020:** Base Salary: $68,000

## Other HR Notes
- **Education:** BS in Computer Science from University of Manchester
- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.
- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.

In [47]:
reranked = rerank(question, chunks)

[17, 2, 5, 4, 12, 15, 1, 13, 7, 6, 9, 3, 11, 18, 10, 8, 16, 19, 20, 14]


In [48]:
for index, c in enumerate(reranked):
    if "manchester" in c.page_content.lower():
        print(index)

0


In [49]:
display(Markdown(reranked[0].page_content))

Compensation and Professional Development

Jessica's base salary has increased from $68,000 in 2020 to $92,000 in 2023, with annual bonuses increasing to $6,000 in 2023. She holds a BS in Computer Science from the University of Manchester and is proficient in React, TypeScript, HTML/CSS, and Jest. She is currently learning Next.js and GraphQL and completed an Advanced React Patterns course in 2023. She also actively contributes to open-source projects.

## Compensation History
- **2023:** Base Salary: $92,000 + Bonus: $6,000
- **2022:** Base Salary: $85,000 + Bonus: $4,000
- **2021:** Base Salary: $72,000 + Bonus: $2,000
- **2020:** Base Salary: $68,000

## Other HR Notes
- **Education:** BS in Computer Science from University of Manchester
- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.
- **Professional Development:** Completed Advanced React Patterns course (2023). Actively contributes to open-source projects.

In [52]:
def fetch_context(question):
    chunks = fetch_context_unranked(question)
    return rerank(question, chunks)

In [53]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
Your answer will be evaluated for accuracy, relevance and completeness, so make sure it only answers the question and fully answers it.
If you don't know the answer, say so.
For context, here are specific extracts from the Knowledge Base that might be directly relevant to the user's question:
{context}

With this context, please answer the user's question. Be accurate, relevant and complete.
"""

In [54]:
# In the context, include the source of the chunk

def make_rag_messages(question, history, chunks):
    context = "\n\n".join(f"Extract from {chunk.metadata['source']}:\n{chunk.page_content}" for chunk in chunks)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    return [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": question}]

In [55]:
def rewrite_query(question, history=[]):
    """Rewrite the user's question to be a more specific question that is more likely to surface relevant content in the Knowledge Base."""
    message = f"""
You are in a conversation with a user, answering questions about the company Insurellm.
You are about to look up information in a Knowledge Base to answer the user's question.

This is the history of your conversation so far with the user:
{history}

And this is the user's current question:
{question}

Respond only with a single, refined question that you will use to search the Knowledge Base.
It should be a VERY short specific question most likely to surface content. Focus on the question details.
Don't mention the company name unless it's a general question about the company.
IMPORTANT: Respond ONLY with the knowledgebase query, nothing else.
"""
    response = completion(model=MODEL, messages=[{"role": "system", "content": message}])
    return response.choices[0].message.content

In [58]:
rewrite_query("Who won the IIOTY award?", [])

'Who won the IIOTY award?'

In [59]:
def answer_question(question: str, history: list[dict] = []) -> tuple[str, list]:
    """
    Answer a question using RAG and return the answer and the retrieved context
    """
    query = rewrite_query(question, history)
    print(query)
    chunks = fetch_context(query)
    messages = make_rag_messages(question, history, chunks)
    response = completion(model=MODEL, messages=messages)
    return response.choices[0].message.content, chunks

In [60]:
answer_question("Who won the IIOTY award?", [])

Who won the IIOTY award?
[1, 2, 14, 20, 9, 7, 3, 11, 6, 8, 10, 13, 12, 16, 4, 5, 15, 19, 18, 17]


('Maxine Thompson won the Insurellm IIOTY (Innovator of the Year) Award in 2023.',
 [Result(page_content='Senior Data Engineer and Leadership (2021-Present)\n\nSince January 2021, Maxine has been a Senior Data Engineer, leading initiatives that improved data retrieval times by 30%. She mentors junior engineers and plays a strategic role at Insurellm, recognized by the IIOTY Innovator Award in 2023.\n\n- **January 2021 - Present**: **Senior Data Engineer**  \n  * Maxine was promoted to Senior Data Engineer after successfully leading a pivotal project that improved data retrieval times by 30%. She now mentors junior engineers and is involved in strategic data initiatives, solidifying her position as a valued asset at Insurellm. She was recognized as Insurellm Innovator of the year in 2023, receiving the prestigious IIOTY 2023 award.  ', metadata={'source': 'knowledge-base/employees/Maxine Thompson.md', 'type': 'employees'}),
  Result(page_content='Additional HR Notes and Future Developme

In [63]:
answer_question("Who went to Manchester University?", [])

Who attended Manchester University?
[18, 1, 2, 5, 16, 17, 20, 19, 11, 12, 9, 13, 6, 3, 7, 8, 10, 14, 15]


('Jessica Liu attended the University of Manchester, where she earned her BS in Computer Science.',
 [Result(page_content="Compensation and Professional Development\n\nJessica's base salary has increased from $68,000 in 2020 to $92,000 in 2023, with annual bonuses increasing to $6,000 in 2023. She holds a BS in Computer Science from the University of Manchester and is proficient in React, TypeScript, HTML/CSS, and Jest. She is currently learning Next.js and GraphQL and completed an Advanced React Patterns course in 2023. She also actively contributes to open-source projects.\n\n## Compensation History\n- **2023:** Base Salary: $92,000 + Bonus: $6,000\n- **2022:** Base Salary: $85,000 + Bonus: $4,000\n- **2021:** Base Salary: $72,000 + Bonus: $2,000\n- **2020:** Base Salary: $68,000\n\n## Other HR Notes\n- **Education:** BS in Computer Science from University of Manchester\n- **Skills:** Proficient in React, TypeScript, HTML/CSS, Jest for testing. Learning Next.js and GraphQL.\n- **Prof

### Assignment 

- Iterate Ed'donners' implementation to be able to beat his evaluation scores
    - Modify Query expansion and re-writing
    - Hierachical RAG
    - HyDE RAG (Generating Hypothetical Documents from user query and use them for retrieval)
- Modify implementation to use Agentic RAG
    - Using tools (for retrieving source files)
    - Using MCP servers



### Major challenge - Your own private knowledge worker

- Create a knowledge worker on your information to boost productivity
    - Assemble all ypur files in 1 place; your personal knowledge Base
    - Vectorize everything in Chroma - your vector datastore

### Advanced ideas to take it to the next level
- If you use Google workspace, use Google's API to read your own docs (Open source models for privacy and security)
- If you use MS Office, use libraries to read Office docs
- Harder - use libraries to connect to your email inbox, and Slack, and more.